# LPLH2 IF Game - Worldstate Reliability Experiment

Runs the **latest versioned LPLH2 worldstate reliability snapshot** on Zork1 for a **1 epoch x 250 step** experiment.

The notebook pulls the latest code from GitHub into the Colab runtime, so you do not need to manually copy updated `lplh2` files into Drive. Drive is still used for large/local artifacts: the Zork1 ROM, the FM LoRA adapter, and saved logs/results.

| Component | Setting |
|---|---|
| Code source | GitHub repo `ogulcaanozen/lplh2-if-agent`, folder `versions/lplh2_worldstate_reliability_patch_2026-07-02/` |
| LLM_a (action generation) | `Qwen/Qwen2.5-14B-Instruct` (HuggingFace, in-process) |
| fm (validate/extract/split) | `Qwen2.5-1.5B-Instruct` + LoRA v4 (`fm_adapter_v4_autoplay_failures/`) |
| LLM_es / aux modules | disabled; summaries/gate/situation/inventory/failure/repetition use LLM_a (`Qwen/Qwen2.5-14B-Instruct`) locally |
| Affordance brainstorming | local LLM_a fallback (`Qwen/Qwen2.5-14B-Instruct`), fed visible objects, inventory, failures, carryover ideas, and active stored situations |
| Run scope | 1 epoch x 250 steps |
| Game | Configured by `GAME_NAME` / `ROM_FILENAME` below (`zork1` by default) |

**Required Drive layout:**
```
MyDrive/lplh/
  fm_adapter_v4_autoplay_failures/ # fm v4 adapter
  games/
    zork1.z5                       # also accepted under games/zork1/zork1.z5 or old IFGames/games paths
  data/                            # created automatically for logs/results
```

**OpenAI optional for this experiment:** auxiliary modules, including affordance brainstorming, use local/Qwen14B unless you explicitly set an OpenAI model.


## 1. Install dependencies

In [ ]:
import os, subprocess, sys, importlib
from importlib.metadata import version, PackageNotFoundError

# Chroma/OTel must be settled before anything imports chromadb, torchao, or
# other libraries that may transitively cache opentelemetry modules.
os.environ.setdefault("ANONYMIZED_TELEMETRY", "FALSE")
os.environ.setdefault("CHROMA_TELEMETRY_IMPL", "none")

REQUIRED = [
    "opentelemetry-api>=1.40.0",
    "opentelemetry-sdk>=1.40.0",
    "transformers>=4.45",
    "peft>=0.12",
    "accelerate>=0.33",
    "torchao>=0.16",
    "jericho>=3.2",
    "chromadb>=0.5",
    "sentence-transformers>=3.0",
    "openai>=1.40",
]

def parse(v):
    head = v.split("+", 1)[0]
    parts = []
    for p in head.split("."):
        try:
            parts.append(int(p))
        except ValueError:
            break
    return tuple(parts)

for pkg in REQUIRED:
    name, _, min_ver = pkg.partition(">=")
    name = name.split("==")[0].strip()
    needs = False
    try:
        cur = version(name)
        if min_ver and parse(cur) < parse(min_ver):
            print(f"Upgrading {name}: {cur} -> >={min_ver}")
            needs = True
    except PackageNotFoundError:
        print(f"Installing {pkg}...")
        needs = True
    if needs:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pkg])

# If a dependency imported OpenTelemetry before pip upgraded it, the runtime can
# hold a stale module even while importlib.metadata reports the new version.
for mod in list(sys.modules):
    if mod == "opentelemetry" or mod.startswith("opentelemetry."):
        del sys.modules[mod]
importlib.invalidate_caches()

print("All required packages ready.")
print("opentelemetry-api:", version("opentelemetry-api"))
print("opentelemetry-sdk:", version("opentelemetry-sdk"))

import opentelemetry.context as otel_context
print("opentelemetry.context:", getattr(otel_context, "__file__", "<unknown>"))
print("has recursion key:", hasattr(otel_context, "_ON_EMIT_RECURSION_COUNT_KEY"))
if not hasattr(otel_context, "_ON_EMIT_RECURSION_COUNT_KEY"):
    raise RuntimeError(
        "OpenTelemetry is installed but the active opentelemetry.context module "
        "is stale/incompatible. Restart the runtime and rerun this cell before "
        "loading LPLH models."
    )

## 2. Mount Drive + set up paths + environment

In [ ]:
import os, sys, subprocess
from pathlib import Path

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
REPO_URL = "https://github.com/ogulcaanozen/lplh2-if-agent.git"
REPO_BRANCH = "main"
VERSION_SUBDIR = Path("versions/lplh2_worldstate_reliability_patch_2026-07-02")
USE_GITHUB_CODE = True

if IS_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    else:
        print("Drive already mounted.")

    DRIVE_ROOT = Path("/content/drive/MyDrive/lplh")
    FM_ADAPTER = DRIVE_ROOT / "fm_adapter_v4_autoplay_failures"

    if USE_GITHUB_CODE:
        REPO_ROOT = Path("/content/lplh2-if-agent")
        if REPO_ROOT.exists():
            print(f"Updating repo at {REPO_ROOT}...")
            subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
        else:
            print(f"Cloning {REPO_URL}...")
            subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
        PROJ_ROOT = REPO_ROOT / VERSION_SUBDIR
    else:
        # Legacy manual-code path, only if you deliberately copied files to Drive.
        REPO_ROOT = DRIVE_ROOT / "IFGames"
        PROJ_ROOT = REPO_ROOT
else:
    # Local fallback: find the repository root from the current working directory.
    cwd = Path.cwd().resolve()
    REPO_ROOT = cwd
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
        REPO_ROOT = REPO_ROOT.parent
    PROJ_ROOT = REPO_ROOT / VERSION_SUBDIR
    DRIVE_ROOT = REPO_ROOT
    FM_ADAPTER = REPO_ROOT / "fm_adapter_v4_autoplay_failures"

assert PROJ_ROOT.exists(), f"Versioned LPLH2 root not found: {PROJ_ROOT}"
assert (PROJ_ROOT / "lplh2" / "__init__.py").exists(), f"lplh2 package missing under {PROJ_ROOT}"
try:
    GIT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    GIT_COMMIT = "unknown"

# Game selection.
GAME_NAME = "zork1"
ROM_FILENAME = "zork1.z5"
GAME_SLUG = GAME_NAME.lower().replace(" ", "_").replace("-", "_")

ROM_CANDIDATES = [
    DRIVE_ROOT / "games" / GAME_NAME / ROM_FILENAME,
    DRIVE_ROOT / "games" / ROM_FILENAME,
    DRIVE_ROOT / "IFGames" / "games" / GAME_NAME / ROM_FILENAME,
    DRIVE_ROOT / "IFGames" / "games" / ROM_FILENAME,
    PROJ_ROOT / "games" / GAME_NAME / ROM_FILENAME,
    PROJ_ROOT / "games" / ROM_FILENAME,
]
ROM_PATH = next((path for path in ROM_CANDIDATES if path.exists()), ROM_CANDIDATES[0])

# Make `import lplh2` use the GitHub-synced versioned snapshot.
sys.path.insert(0, str(PROJ_ROOT))

# Wire config via env vars (consumed by lplh2.config)
os.environ["LPLH_LLM_PROVIDER"] = "huggingface"
os.environ["LPLH_LLM_MODEL"] = "Qwen/Qwen2.5-14B-Instruct"
os.environ["LPLH_FM_PATH"] = str(FM_ADAPTER)
os.environ["LPLH_FM_BASE"] = "Qwen/Qwen2.5-1.5B-Instruct"
os.environ["LPLH_FM_TEMPERATURE"] = "0.1"
os.environ["LPLH_PERSIST_ACTION_SPACE"] = "true"
os.environ["LPLH_LLM_ES_MODEL"] = ""  # empty => non-brainstorm aux modules use local LLM_a/Qwen14B
os.environ["LPLH_BRAINSTORM_MODEL"] = ""  # empty => affordance brainstorming uses local LLM_a/Qwen14B
os.environ["LPLH_BRAINSTORM_REASONING_EFFORT"] = ""

# OpenAI API key is optional here; it is unused unless an OpenAI aux/brainstorm model is explicitly set.
if IS_COLAB:
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
    except Exception:
        pass

import torch
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"CODE_ROOT   : {PROJ_ROOT}")
print(f"GIT COMMIT  : {GIT_COMMIT}")
print(f"PACKAGE     : lplh2")
print(f"GAME        : {GAME_NAME}")
print(f"FM_ADAPTER  : {FM_ADAPTER}  exists={FM_ADAPTER.exists()}")
print(f"ROM         : {ROM_PATH}    exists={ROM_PATH.exists()}")
print(f"GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB" if torch.cuda.is_available() else "")
print(f"BRAINSTORM  : {os.environ.get('LPLH_BRAINSTORM_MODEL') or 'LLM_a fallback'}")
print(f"BRAIN effort: {os.environ.get('LPLH_BRAINSTORM_REASONING_EFFORT') or 'n/a'}")
print(f"OPENAI key  : {'set' if os.environ.get('OPENAI_API_KEY') else 'MISSING'} (optional; unused unless an OpenAI aux model is set)")
if os.environ.get("LPLH_BRAINSTORM_MODEL") and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required because LPLH_BRAINSTORM_MODEL is set.")

checked_paths = "\n".join(f"  {path}" for path in ROM_CANDIDATES)
assert FM_ADAPTER.exists(), f"fm adapter not at {FM_ADAPTER}"
assert ROM_PATH.exists(), f"{GAME_NAME} ROM not found. Checked:\n{checked_paths}"


## 3. Run Scope

This experiment is configured for **1 epoch x 250 steps** using the GitHub-synced LPLH2 worldstate reliability snapshot. Logs include KG location resolution, inventory reconciliation, dynamic situation decisions, affordance brainstorming, and per-module timing.


In [ ]:
from datetime import datetime

# Main evaluation run for the latest worldstate reliability LPLH2 version.
NUM_EPOCHS = 1
MAX_STEPS = 250
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Run scope: {NUM_EPOCHS} epoch(s) x {MAX_STEPS} steps")
print(f"Run id   : {RUN_ID}")


## 4. Load models (LLM_a + fm) - one-time, ~3 min

Both Qwen2.5-14B-Instruct (LLM_a, ~28 GB bf16) and Qwen2.5-1.5B + LoRA adapter (fm, ~3 GB) load into A100's 40 GB VRAM. Restart the runtime if CUDA memory is fragmented.

In [ ]:
import sys, os, importlib, importlib.util

# Robust import setup - handles Drive sync lag + sys.modules cache poisoning.
PROJ_ROOT_STR = str(PROJ_ROOT)
if PROJ_ROOT_STR not in sys.path:
    sys.path.insert(0, PROJ_ROOT_STR)

# Purge stale package entries from earlier notebook attempts.
for mod in list(sys.modules):
    if mod == "lplh2" or mod.startswith("lplh2."):
        del sys.modules[mod]
for k in list(sys.modules):
    if sys.modules[k] is None:
        del sys.modules[k]

importlib.invalidate_caches()

spec = importlib.util.find_spec("lplh2")
if spec is None:
    raise ImportError(
        f"Cannot locate `lplh2` package. Checked sys.path:\\n"
        + "\\n".join(f"  {p}" for p in sys.path[:5])
        + f"\\n\\nVerify {PROJ_ROOT_STR}/lplh2/__init__.py exists on Drive."
    )
print(f"lplh2 package located at: {spec.origin}")

from lplh2.llm_client import LLMClient
from lplh2.fm_client import FmClient

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. HF model loading requires a GPU runtime.")
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu_name = torch.cuda.get_device_name(0)
min_vram_gb = 38.0
print(f"GPU check   : {gpu_name} ({total_vram_gb:.1f} GB VRAM)")
if total_vram_gb < min_vram_gb and os.environ.get("LPLH_ALLOW_LOW_VRAM") != "1":
    raise RuntimeError(
        f"This runtime has {total_vram_gb:.1f} GB VRAM, which is too small for "
        "unquantized Qwen2.5-14B + fm v4 loading. Use an A100/High-RAM Colab "
        "runtime or a >=40 GB GPU box. If you deliberately want to risk CPU/disk "
        "offload, set os.environ['LPLH_ALLOW_LOW_VRAM']='1' before this cell."
    )

print("\\nLoading LLM_a (Qwen2.5-14B-Instruct)...")
llm = LLMClient()
print("\\nLoading fm v4 (Qwen2.5-1.5B + LoRA)...")
fm = FmClient()

free, total = torch.cuda.mem_get_info()
print(f"\\nVRAM in use: {(total - free)/1024**3:.1f} / {total/1024**3:.1f} GB")


## 5. Quick sanity check on each fm task before the full run

In [ ]:
# Validate
v_true = fm.validate_action("north", "North of House. You are facing the north side of a white house.")
v_false = fm.validate_action("xyzzy", "I don't know the word 'xyzzy'.")
v_mailbox = fm.validate_action("take mailbox", "It is securely anchored.")
v_grating = fm.validate_action("take grating", "An interesting idea...")
v_impenetrable = fm.validate_action("north", "The forest becomes impenetrable to the north.")
print(f"fm.validate_action ('north' -> success):   {v_true}")
print(f"fm.validate_action ('xyzzy' -> failure):   {v_false}")
print(f"fm.validate_action ('take mailbox' anchored): {v_mailbox}  # expected False")
print(f"fm.validate_action ('take grating' idea):     {v_grating}  # expected False")
print(f"fm.validate_action ('north' impenetrable):    {v_impenetrable}  # expected False")

# Extract
triples = fm.extract_relations("look", "West of House. You are standing in an open field west of a white house, with a boarded front door. There is a small mailbox here.")
print(f"\nfm.extract_relations:")
for t in triples:
    print(f"  <{t[0]}, {t[1]}, {t[2]}>")

# Split
print(f"\nfm.split_action('U'): {fm.split_action('U')}")
print(f"fm.split_action('put sword in box'): {fm.split_action('put sword in box')}")

# LLM_a probe
resp = llm.chat("You are an expert IF player.", "Reply with exactly one word: ok", temperature=0.0)
print(f"\nLLM_a probe: {resp[:80]!r}")

## 6. Run the game

This run keeps the lplh2 runtime path and auxiliary mechanisms enabled: FM validation/extraction, robust command parsing, summary storage, stored situations, environmental-change detection, affordance brainstorming/carryover, failed-command memory, and main LLM reasoning.


In [ ]:
from pathlib import Path
from lplh2.agent import LPLHAgent
from lplh2.game_runner import GameRunner
from lplh2 import config as lplh2_config

# Override config knobs for this run
lplh2_config.NUM_EPOCHS = NUM_EPOCHS
lplh2_config.MAX_STEPS_PER_EPOCH = MAX_STEPS

# Make data + logs go to Drive (Colab) so they survive runtime disconnects.
# Chroma and the neutral-event dedup index get fresh per-run directories so
# experiences persist across epochs within this run but cannot leak in from
# earlier lplh/lplh2 experiments.
if IS_COLAB:
    lplh2_config.DATA_DIR = str(DRIVE_ROOT / "data")
    lplh2_config.LOGS_DIR = str(DRIVE_ROOT / "data" / "logs")
    lplh2_config.CHROMA_PERSIST_DIR = str(DRIVE_ROOT / "data" / "chroma_db" / f"lplh2_{GAME_SLUG}_{RUN_ID}")
    lplh2_config.EXPERIENCE_INDEX_DIR = str(DRIVE_ROOT / "data" / "experience_index" / f"lplh2_{GAME_SLUG}_{RUN_ID}")
    Path(lplh2_config.LOGS_DIR).mkdir(parents=True, exist_ok=True)
else:
    lplh2_config.CHROMA_PERSIST_DIR = str(PROJ_ROOT / "data" / "chroma_db" / f"lplh2_{GAME_SLUG}_{RUN_ID}")
    lplh2_config.EXPERIENCE_INDEX_DIR = str(PROJ_ROOT / "data" / "experience_index" / f"lplh2_{GAME_SLUG}_{RUN_ID}")
print(f"Chroma DB   : {lplh2_config.CHROMA_PERSIST_DIR}")
print(f"Exp index   : {lplh2_config.EXPERIENCE_INDEX_DIR}")
print(f"fm temp     : {lplh2_config.FM_TEMPERATURE}")
print(f"AS persists : {lplh2_config.PERSIST_ACTION_SPACE}")

# Monkey-patch LLMClient/FmClient inside game_runner's namespace so GameRunner.run()
# uses our PRELOADED instances instead of re-loading the 28 GB Qwen2.5-14B + 3 GB fm.
import lplh2.game_runner as gr_mod
_orig_LLMClient = gr_mod.LLMClient
_orig_FmClient = gr_mod.FmClient
gr_mod.LLMClient = lambda *a, **kw: llm
gr_mod.FmClient = lambda *a, **kw: fm

runner = GameRunner(
    game_path=str(ROM_PATH),
    num_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    verbose=True,
)
try:
    results = runner.run()
finally:
    gr_mod.LLMClient = _orig_LLMClient
    gr_mod.FmClient = _orig_FmClient


## 7. Report + saved log artifacts

This cell prints the Drive paths for the full human-readable run log, detailed JSON step log, result JSON, and a compact summary file. Run it even if you interrupted the game cell.


In [ ]:
from collections import Counter
import json, re
from pathlib import Path

# Results may be partial if the run was interrupted before an epoch completed.
try:
    results
except NameError:
    results = {}
runner_obj = globals().get("runner", None)
epoch_results = results.get("all_epoch_results", results.get("epoch_results", [])) if isinstance(results, dict) else []
scores = [er.get("final_score", 0) for er in epoch_results]

run_log_path = Path(results.get("run_log_path") or getattr(runner_obj, "_run_log_path", ""))
step_log_path = Path(results.get("step_log_path") or getattr(runner_obj, "_step_log_path", ""))
summary_log_path = Path(results.get("summary_log_path") or getattr(runner_obj, "_summary_log_path", ""))
situation_log_path = Path(results.get("situation_log_path") or getattr(runner_obj, "_situation_log_path", ""))
affordance_log_path = Path(results.get("affordance_log_path") or getattr(runner_obj, "_affordance_log_path", ""))
action_failure_log_path = Path(results.get("action_failure_log_path") or getattr(runner_obj, "_action_failure_log_path", ""))
action_generation_log_path = Path(results.get("action_generation_log_path") or getattr(runner_obj, "_action_generation_log_path", ""))
auxiliary_gate_log_path = Path(results.get("auxiliary_gate_log_path") or getattr(runner_obj, "_auxiliary_gate_log_path", ""))
results_path = Path(results.get("results_path") or getattr(runner_obj, "_results_path", ""))
log_dir_path = Path(results.get("log_dir_path") or getattr(runner_obj, "_experiment_log_dir", "") or run_log_path.parent or lplh2_config.LOGS_DIR)
summary_path = log_dir_path / "summary.txt"

payload = {}
epoch_logs = []
if step_log_path and step_log_path.exists():
    payload = json.loads(step_log_path.read_text(encoding="utf-8"))
    if isinstance(payload, dict):
        epoch_logs = payload.get("epochs", [])
    elif isinstance(payload, list):  # backward-compatible with older steplogs
        epoch_logs = payload

steps = []
for ep in epoch_logs:
    for step in ep.get("steps", []):
        step = dict(step)
        step["epoch"] = ep.get("epoch")
        steps.append(step)

def extract_reason(raw):
    if not raw:
        return ""
    m = re.search(r"<rea>(.*?)</rea>", str(raw), re.DOTALL | re.IGNORECASE)
    if m:
        return re.sub(r"\s+", " ", m.group(1)).strip()
    return ""

neutral_stored_counts = Counter()
neutral_skipped_counts = Counter()
retrieved_steps = 0
score_experiences = 0
situation_status_counts = Counter()
situation_resolution_status_counts = Counter()
situations_removed_total = 0
environmental_status_counts = Counter()
environmental_changes_detected = 0
affordance_status_counts = Counter()
action_failure_status_counts = Counter()
auxiliary_gate_status_counts = Counter()
auxiliary_gate_summary_trigger_counts = Counter()
auxiliary_gate_affordance_run = 0
auxiliary_gate_legacy_summary_fallback = 0
action_failures_stored = 0
action_failures_removed = 0
affordance_idea_groups = 0
affordance_carryover_steps = 0
affordance_filtered_failed_total = 0
last_situations = []
last_verbs = []
last_rooms = []
recent_reasoning = []
recent_brainstorm_reasons = []

for step in steps:
    modules = step.get("modules", {})
    exp = modules.get("experience_lib", {})
    kg = modules.get("kg_map", {})
    act = modules.get("action_space", {})
    gen = modules.get("action_generation", {})
    sit = modules.get("situation_memory", {})
    aff = gen.get("affordance_brainstorming", {})
    fail_mem = modules.get("action_failure_memory", {})
    env = modules.get("environmental_change_detection", {})
    gate = modules.get("auxiliary_gate", {})

    if gate.get("status"):
        auxiliary_gate_status_counts[gate.get("status")] += 1
    if gate.get("use_legacy_summary_trigger_detection"):
        auxiliary_gate_legacy_summary_fallback += 1
    gate_decision = gate.get("decision", {}) if isinstance(gate.get("decision", {}), dict) else {}
    for trigger_name, trigger_decision in (gate_decision.get("summary_triggers", {}) or {}).items():
        if isinstance(trigger_decision, dict) and trigger_decision.get("run"):
            auxiliary_gate_summary_trigger_counts[trigger_name] += 1
    aff_gate = gate_decision.get("affordance_brainstorming", {})
    if isinstance(aff_gate, dict) and aff_gate.get("run"):
        auxiliary_gate_affordance_run += 1

    if sit.get("status"):
        situation_status_counts[sit.get("status")] += 1
    resolution = sit.get("resolution", {})
    if resolution.get("status"):
        situation_resolution_status_counts[resolution.get("status")] += 1
    situations_removed_total += len(resolution.get("removed_situations", []))
    if sit.get("active_situations_after"):
        last_situations = sit.get("active_situations_after", [])
    if aff.get("status"):
        affordance_status_counts[aff.get("status")] += 1
    affordance_idea_groups += len(aff.get("ideas", []))
    if aff.get("carried_ideas_before"):
        affordance_carryover_steps += 1
    affordance_filtered_failed_total += len(aff.get("filtered_failed_commands", []))
    for idea in aff.get("ideas", [])[:3]:
        reason_text = idea.get("reason", "")
        if reason_text:
            recent_brainstorm_reasons.append((
                step.get("epoch"),
                step.get("step"),
                aff.get("status"),
                idea.get("situation", ""),
                ", ".join(idea.get("commands_to_try", [])),
                reason_text,
            ))
    if env.get("status"):
        environmental_status_counts[env.get("status")] += 1
    if env.get("environmental_change"):
        environmental_changes_detected += 1
    if fail_mem.get("status"):
        action_failure_status_counts[fail_mem.get("status")] += 1
    if fail_mem.get("status") == "stored":
        action_failures_stored += 1
    if fail_mem.get("removed_failure"):
        action_failures_removed += 1

    if exp.get("retrieved_experiences") and exp.get("retrieved_experiences") != "No relevant experiences found yet.":
        retrieved_steps += 1
    if exp.get("experience_triggered", exp.get("score_changed")) and exp.get("new_experience_summary"):
        score_experiences += 1
    for trigger, _summary in exp.get("neutral_summaries", []):
        neutral_stored_counts[trigger] += 1
    for skipped in exp.get("neutral_summaries_skipped", []):
        neutral_skipped_counts[skipped.get("trigger", "neutral")] += 1
    if act.get("all_verbs"):
        last_verbs = sorted(act.get("all_verbs", []))
    if kg.get("rooms_visited"):
        last_rooms = kg.get("rooms_visited", [])

    for label, gen_payload, command in [
        ("executed", gen, step.get("final_command")),
        ("next", modules.get("next_action_generation", {}), step.get("next_command")),
    ]:
        reason = extract_reason(gen_payload.get("llm_raw_response", ""))
        if reason:
            recent_reasoning.append((step.get("epoch"), step.get("step"), label, command, reason))

lines = []
lines.append("=" * 72)
lines.append(f"LPLH2 WORLDSTATE RELIABILITY EXPERIMENT REPORT - {GAME_NAME}")
lines.append("=" * 72)
lines.append(f"Run id              : {RUN_ID}")
lines.append(f"Run complete/partial: {'partial/interrupted' if results.get('partial') else 'completed'}")
lines.append(f"Saved step count    : {len(steps)}")
if scores:
    last_3_avg = sum(scores[-3:]) / max(1, min(3, len(scores)))
    lines.append(f"Per-epoch scores    : {scores}")
    lines.append(f"Avg last 3          : {last_3_avg:.1f}")
    lines.append(f"Run max             : {max(scores)}")
else:
    lines.append("Per-epoch scores    : none completed")
lines.append("")
lines.append(f"Run log txt         : {run_log_path}")
lines.append(f"Step log json       : {step_log_path}")
lines.append(f"Summary module log : {summary_log_path}")
lines.append(f"Situation memory log: {situation_log_path}")
lines.append(f"Affordance log     : {affordance_log_path}")
lines.append(f"Action failure log : {action_failure_log_path}")
lines.append(f"Action gen log     : {action_generation_log_path}")
lines.append(f"Auxiliary gate log : {auxiliary_gate_log_path}")
lines.append(f"Experiment log dir  : {log_dir_path}")
lines.append(f"Results json        : {results_path}")
lines.append(f"Chroma DB           : {lplh2_config.CHROMA_PERSIST_DIR}")
lines.append(f"Experience index    : {lplh2_config.EXPERIENCE_INDEX_DIR}")
lines.append("")
lines.append(f"Rooms seen          : {', '.join(last_rooms) if last_rooms else '(none)'}")
lines.append(f"Learned verbs       : {', '.join(last_verbs) if last_verbs else '(none)'}")
lines.append(f"Retrieved on steps  : {retrieved_steps}")
lines.append(f"Score summaries     : {score_experiences}")
lines.append("Neutral stored      : " + (", ".join(f"{k}={v}" for k, v in sorted(neutral_stored_counts.items())) if neutral_stored_counts else "none"))
lines.append("Neutral skipped dup : " + (", ".join(f"{k}={v}" for k, v in sorted(neutral_skipped_counts.items())) if neutral_skipped_counts else "none"))
lines.append("Situation statuses : " + (", ".join(f"{k}={v}" for k, v in sorted(situation_status_counts.items())) if situation_status_counts else "none"))
lines.append("Situation resolutions: " + (", ".join(f"{k}={v}" for k, v in sorted(situation_resolution_status_counts.items())) if situation_resolution_status_counts else "none"))
lines.append(f"Situations removed : {situations_removed_total}")
lines.append("Environmental detector: " + (", ".join(f"{k}={v}" for k, v in sorted(environmental_status_counts.items())) if environmental_status_counts else "none"))
lines.append(f"Environmental changes: {environmental_changes_detected}")
lines.append("Aux gate statuses  : " + (", ".join(f"{k}={v}" for k, v in sorted(auxiliary_gate_status_counts.items())) if auxiliary_gate_status_counts else "none"))
lines.append("Aux gate summary triggers: " + (", ".join(f"{k}={v}" for k, v in sorted(auxiliary_gate_summary_trigger_counts.items())) if auxiliary_gate_summary_trigger_counts else "none"))
lines.append(f"Aux gate affordance run decisions: {auxiliary_gate_affordance_run}")
lines.append(f"Aux gate legacy summary fallback: {auxiliary_gate_legacy_summary_fallback}")
lines.append("Affordance statuses: " + (", ".join(f"{k}={v}" for k, v in sorted(affordance_status_counts.items())) if affordance_status_counts else "none"))
lines.append(f"Affordance ideas   : {affordance_idea_groups}")
lines.append(f"Affordance carryover steps: {affordance_carryover_steps}")
lines.append(f"Filtered failed cmds in affordance: {affordance_filtered_failed_total}")
lines.append("Action failure statuses: " + (", ".join(f"{k}={v}" for k, v in sorted(action_failure_status_counts.items())) if action_failure_status_counts else "none"))
lines.append(f"Action failures stored : {action_failures_stored}")
lines.append(f"Action failures removed: {action_failures_removed}")
lines.append("Active situations   : " + (json.dumps(last_situations, ensure_ascii=False) if last_situations else "none"))
lines.append("")
lines.append("Recent brainstorm reasons:")
for epoch, step_no, status, situation, commands, reason in recent_brainstorm_reasons[-10:]:
    lines.append(f"  E{epoch} S{step_no} [{status}] {situation[:90]} -> {commands[:120]} | reason: {reason[:220]}")
if not recent_brainstorm_reasons:
    lines.append("  (none extracted)")
lines.append("")
lines.append("Recent main LLM reasoning snippets:")
for epoch, step_no, label, command, reason in recent_reasoning[-10:]:
    lines.append(f"  E{epoch} S{step_no} {label} cmd={command!r}: {reason[:240]}")
if not recent_reasoning:
    lines.append("  (none extracted)")
lines.append("")
lines.append("Full details are in the run log text, JSON step log, summary module log, situation memory log, auxiliary gate log (`auxiliary_gate_log.txt`), affordance brainstorm log, action failure memory log, and action generation log. The auxiliary gate log contains routing prompts/responses, summary-trigger decisions, and gate fallback status. The summary module log contains the state type, exact summarization prompt, and generated summary for each newly stored summary. The situation memory log contains the detector prompt, raw LLM response, parsed situation, and all active stored situations plus resolution decisions after every step. The affordance log contains the current observation, affordance agenda, pending carryover commands, tried/failed context, brainstorm prompt, raw response, parsed command ideas, carryover before/after, and filtered failed commands. The action failure log contains failure-memory status, world signature, failure-reason prompt/response, stored record, removed record, and known failures after each step. The action generation log contains the main LLM raw responses, extracted reasoning, selected commands, retrieved experiences, affordance agenda, known failures, and action-space context.")

summary_text = "\n".join(lines)
summary_path.write_text(summary_text, encoding="utf-8")
print(summary_text)
print(f"\nCompact summary saved to: {summary_path}")
